# Track B — MASt3R at 1024 px (EXPERIMENTAL, NOT SUBMITTED)

We tried running MASt3R at 1024-pixel input resolution to see if higher resolution would yield finer geometric detail. **The results were visibly distorted for the Fountain scene** (and we expect similar for the other datasets), so we reverted to the 512 px version for our final submission.

This notebook is preserved for reproducibility of the experiment. The two relevant differences from the 512 px notebook are:

1. `load_images(imgs, size=1024)` instead of `size=512`.
2. The confidence-value distribution shifts at 1024 px, so a fixed `CONF_THRESH = 1.7` masks away every point. We replaced the fixed threshold with an **adaptive percentile-based threshold** (keep the top `KEEP_FRAC` of points by confidence).

**Required runtime**: Colab with **A100 GPU** and **High-RAM**.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone MASt3R and install dependencies

In [ ]:
import os
os.chdir('/content')
if not os.path.exists('mast3r'):
    !git clone --recursive https://github.com/naver/mast3r.git
os.chdir('/content/mast3r')
!pip install -q -r requirements.txt
!pip install -q -r dust3r/requirements.txt
!pip install -q open3d trimesh

## 3. Download MASt3R weights

In [ ]:
import os
os.makedirs('/content/mast3r/checkpoints', exist_ok=True)
!wget -nc -O /content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth

## 4. Mount Google Drive and load images (Option B)

In [ ]:
from google.colab import drive
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# >>> CHANGE BETWEEN RUNS <<<
DATASET_NAME = 'Fountain'

DATA_DIR = f'/content/drive/MyDrive/stem_games_data/{DATASET_NAME}'

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f'Folder not found: {DATA_DIR}')

imgs = sorted([
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])
if not imgs:
    raise RuntimeError(f'No images found in {DATA_DIR}')

print(f'Dataset: {DATASET_NAME}')
print(f'{len(imgs)} images ready')

## 5. Run MASt3R inference at 1024 px

**Note**: 1024 px inference is ~4× slower than 512 px and uses substantially more VRAM. Make sure you're on A100 + High-RAM.

In [ ]:
import sys
sys.path.insert(0, '/content/mast3r')
sys.path.insert(0, '/content/mast3r/dust3r')

import torch
from mast3r.model import AsymmetricMASt3R
from dust3r.inference import inference
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

device = 'cuda'
ckpt = '/content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
model = AsymmetricMASt3R.from_pretrained(ckpt).to(device)

# *** DIFFERENCE FROM 512 NOTEBOOK: size=1024 ***
images = load_images(imgs, size=1024)
print(f'Loaded {len(images)} images at 1024 px')

pairs = make_pairs(images, scene_graph='complete', prefilter=None, symmetrize=True)
print(f'{len(pairs)} pairs')

with torch.no_grad():
    output = inference(pairs, model, device, batch_size=1, verbose=True)

scene = global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
loss = scene.compute_global_alignment(init='mst', niter=300, schedule='cosine', lr=0.01)
print(f'Final alignment loss: {loss}')

## 6. Export with adaptive confidence threshold

Confidence values at 1024 px land in a different numerical range than at 512 px, so the fixed threshold of 1.7 we used at 512 px would mask away **every** point at 1024 px. This cell picks the threshold adaptively from the actual confidence distribution — keep the top `KEEP_FRAC` fraction of points by confidence.

In [ ]:
import numpy as np

# === Parameters ===
VOX = 0.001
KEEP_FRAC = 0.80   # keep the top 80% of points by confidence
                   # 0.95 = very permissive; 0.50 = very strict

pts3d  = scene.get_pts3d()
imgs_t = scene.imgs
confs  = scene.get_conf()

all_p, all_c, all_rgb = [], [], []
for p, c, im in zip(pts3d, confs, imgs_t):
    all_p.append(p.detach().cpu().numpy().reshape(-1, 3))
    all_c.append(c.detach().cpu().numpy().reshape(-1))
    all_rgb.append((np.asarray(im) * 255).astype(np.uint8).reshape(-1, 3))
P_all   = np.concatenate(all_p, axis=0)
C_all   = np.concatenate(all_c, axis=0)
RGB_all = np.concatenate(all_rgb, axis=0)

# Diagnostic: confidence distribution
print(f'Confidence stats:')
print(f'  min={C_all.min():.3f}  max={C_all.max():.3f}  mean={C_all.mean():.3f}')
print(f'  percentiles: 10%={np.percentile(C_all,10):.3f}  '
      f'50%={np.percentile(C_all,50):.3f}  '
      f'90%={np.percentile(C_all,90):.3f}')

conf_thresh = np.percentile(C_all, 100 * (1 - KEEP_FRAC))
print(f'\nAdaptive conf_thresh = {conf_thresh:.3f} (keeps top {KEEP_FRAC*100:.0f}%)')

mask = C_all > conf_thresh
P = P_all[mask].astype(np.float32)
C = RGB_all[mask].astype(np.uint8)
print(f'Total points: {len(P):,}')

# Voxel dedup
coords = np.floor(P / VOX).astype(np.int64)
key = coords[:,0]*1_000_003*1_000_003 + coords[:,1]*1_000_003 + coords[:,2]
order = np.argsort(key, kind='stable')
key_s, P_s, C_s = key[order], P[order], C[order]
starts = np.concatenate([[0], np.where(np.diff(key_s) != 0)[0] + 1])
ends   = np.concatenate([starts[1:], [len(key_s)]])
P_dd = np.stack([P_s[s:e].mean(0) for s, e in zip(starts, ends)])
C_dd = np.stack([C_s[s:e].mean(0).astype(np.uint8) for s, e in zip(starts, ends)])
print(f'After voxel-dedup: {len(P_dd):,}')

# PLY
out_ply = f'/content/{DATASET_NAME.lower()}_track_b_1024.ply'
with open(out_ply, 'wb') as f:
    header = (f'ply\nformat binary_little_endian 1.0\nelement vertex {len(P_dd)}\n'
              f'property float x\nproperty float y\nproperty float z\n'
              f'property uchar red\nproperty uchar green\nproperty uchar blue\n'
              f'end_header\n').encode('ascii')
    f.write(header)
    dt = np.dtype([('x','<f4'),('y','<f4'),('z','<f4'),('r','u1'),('g','u1'),('b','u1')])
    arr = np.empty(len(P_dd), dtype=dt)
    arr['x'], arr['y'], arr['z'] = P_dd[:,0], P_dd[:,1], P_dd[:,2]
    arr['r'], arr['g'], arr['b'] = C_dd[:,0], C_dd[:,1], C_dd[:,2]
    f.write(arr.tobytes())
print(f'Wrote {out_ply}')

out_txt = f'/content/{DATASET_NAME.lower()}_track_b_1024.txt'
np.savetxt(out_txt,
           np.concatenate([P_dd, C_dd.astype(np.int32)], axis=1),
           fmt=['%.4f','%.4f','%.4f','%d','%d','%d'])
print(f'Wrote {out_txt}')

## 7. Download files

Save the 1024 outputs separately so you can compare against the 512 results.

In [ ]:
from google.colab import files
files.download(out_ply)
files.download(out_txt)